In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
%%bash
# Uninstall everything first
pip uninstall wfdb pandas -y

# Install a stable Pandas 2.x version
pip install pandas==2.3.3

# Now install wfdb 4.2.0 WITHOUT upgrading Pandas
pip install wfdb==4.2.0 --no-deps

Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.3/162.3 kB 3.5 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
dask-cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 require

In [3]:
%%bash
pip uninstall wandb -y
pip cache purge
pip install wandb==0.23.1

Found existing installation: wandb 0.22.2
Uninstalling wandb-0.22.2:
  Successfully uninstalled wandb-0.22.2
Files removed: 12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.9/22.9 MB 71.9 MB/s eta 0:00:00


In [4]:
import torch.nn as nn
import torch
import torch.nn.functional as F
from torch.optim import Adam,AdamW
from torch.utils.data import DataLoader, random_split
from typing import Tuple,Dict
from torch.utils.data import Dataset
import numpy as np
import os
import random
import wfdb
import wandb


In [5]:
dataset_path = os.path.join('/kaggle','input','datasets','yasanthaniroshan','irida-200-rr')
df = pd.read_csv(os.path.join(dataset_path,'200x20_extracted_rr_intervals.csv'))

In [6]:
df.head()

,patient_id,episode_id,segment_id,EventType,Event,TimeToEvent,RMSSD,pNN50,SDNN,alpha_1,sample_entropy,approximate_entropy
0,record_117,0,0,NSR,1,3403.825,0.388317,94.472362,0.285732,0.763981,0.725235,0.717615
1,record_117,0,1,NSR,1,3387.465,0.382623,93.467337,0.286787,0.585782,0.699762,0.702439
2,record_117,0,2,NSR,1,3371.865,0.378015,93.467337,0.284531,0.438095,0.739817,0.717142
3,record_117,0,3,NSR,1,3356.115,0.373955,92.462312,0.282570,0.532514,0.769174,0.719268
4,record_117,0,4,NSR,1,3338.115,0.357152,91.457286,0.274665,0.618606,0.759568,0.730037


In [7]:
import os
os.environ['WANDB_API_KEY'] = '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'

In [8]:
!wandb login

wandb: Currently logged in as: yasantha-21 (eml-labs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [9]:
def compute_hrv_metrics(rr_window):
    rr_diff = np.diff(rr_window)
    rmssd = np.sqrt(np.mean(rr_diff ** 2)) if len(rr_diff) else 0.0
    sdnn = np.std(rr_window)
    sd1 = np.std(rr_diff) / np.sqrt(2) if len(rr_diff) else 0.0
    sd2_val = 2 * sdnn ** 2 - sd1 ** 2
    sd2 = np.sqrt(sd2_val) if sd2_val > 0 else 0.0

    hrv = np.array([rmssd, sdnn, sd1, sd2], dtype=np.float32)
    return np.log1p(hrv * 10)

In [10]:
random.seed(42)
torch.random.manual_seed(42)

In [11]:
# Hyperparameters
LR = 2e-4
EPOCHS = 100
LATENT_SIZE = 32
CONTEXT_SIZE = 64
NUMBER_OF_TARGETS_FOR_PREDICTION = 4
WINDOW_SIZE = 200
STRIDE = 10
HORIZONS = [4,8,16]
SEQUENCE_LENGTH = 20
VALIDATION_RATIO = 0.2
BATCH_SIZE = 256
PREDICTION = "hrv difference"

In [12]:
# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="eml-labs",
    # Set the wandb project where this run will be logged.
    project="Prediction-PAF-Onset-using-CNN-sliding-window-using-RR-intervals",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": LR,
        "epochs": EPOCHS,
        "latent_size": LATENT_SIZE,
        "context_size": CONTEXT_SIZE,
        "number_of_targets_for_prediction": NUMBER_OF_TARGETS_FOR_PREDICTION,
        "window_size": WINDOW_SIZE,
        "stride": STRIDE,
        "horizons": HORIZONS,
        "sequence_length": SEQUENCE_LENGTH,
        "validation_ratio": VALIDATION_RATIO,
        "batch_size": BATCH_SIZE,
        "prediction": PREDICTION,
        "dataset": "IRIDA-AF",
    },
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Currently logged in as: yasantha-21 (eml-labs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [13]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=LATENT_SIZE):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        self.proj = nn.Linear(32, latent_dim)

    def forward(self, rr_window)->torch.Tensor:
        """
        rr_window: [B, 50]
        output:    [B, 32]
        """
        x = rr_window.unsqueeze(1)
        h = self.encoder(x).squeeze(-1)
        z = self.proj(h)
        return z


In [14]:
class ARBlock(nn.Module):
    def __init__(self, latent_dim=LATENT_SIZE, context_dim=CONTEXT_SIZE):
        super().__init__()

        self.gru = nn.GRU(
            input_size=latent_dim,
            hidden_size=context_dim,
            batch_first=True
        )

    def forward(self, z_seq)->torch.Tensor:
        """
        z_seq: [B, T, latent_dim]
        output: [B, T, context_dim]
        """
        c_seq, _ = self.gru(z_seq)
        return c_seq   # [B, T, context_dim]


In [15]:
class HRVPredictor(nn.Module):
    def __init__(self, context_dim=CONTEXT_SIZE, num_targets=NUMBER_OF_TARGETS_FOR_PREDICTION):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(context_dim, 16),
            nn.ReLU(),
            nn.Linear(16, num_targets)
        )

    def forward(self, c_t)->torch.Tensor:
        """
        c_t: [B, context_dim]
        output: [B, num_targets]
        """
        return self.net(c_t)

In [16]:
class MultiStepHRVPredictor(nn.Module):
    def __init__(self, context_dim=CONTEXT_SIZE, num_targets=NUMBER_OF_TARGETS_FOR_PREDICTION):
        super().__init__()
        self.predictor_1 = HRVPredictor(context_dim, num_targets)
        self.predictor_2 = HRVPredictor(context_dim, num_targets)
        self.predictor_4 = HRVPredictor(context_dim, num_targets)

    def forward(self, c_t:torch.Tensor,last_rr)->Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        c_t: [B, context_dim]
        returns:
          y1, y2, y4: [B, num_targets] each
        """
        y1 = self.predictor_1(c_t) 
        y2 = self.predictor_2(c_t)
        y4 = self.predictor_4(c_t)
        return y1, y2, y4

In [17]:
class CPCPreModel(nn.Module):
    def __init__(self, num_targets=NUMBER_OF_TARGETS_FOR_PREDICTION):
        super().__init__()
        self.encoder = Encoder(latent_dim=LATENT_SIZE)
        self.context = ARBlock(latent_dim=LATENT_SIZE, context_dim=CONTEXT_SIZE)
        self.predictor = MultiStepHRVPredictor(context_dim=CONTEXT_SIZE, num_targets=num_targets)

    def forward(self, rr_windows: torch.Tensor) -> torch.Tensor:
        """
        rr_windows: [B, T, W] 
        Returns:
            c_seq: [B, T, context_dim]
        """
        
        B, T, W = rr_windows.shape
        z_list = []

        for t in range(T):
            z_t = self.encoder(rr_windows[:, t, :])  # [B, latent_dim]
            z_list.append(z_t)

        z_seq = torch.stack(z_list, dim=1)  # [B, T, latent_dim]
        c_seq = self.context(z_seq)         # [B, T, context_dim]

        return c_seq

In [18]:
class RRSequenceDataset(Dataset):
    def __init__(self,dataset_path,csv_file_name,patient_list, window_size=WINDOW_SIZE, stride=STRIDE, horizons=HORIZONS, seq_len=SEQUENCE_LENGTH):

        self.window_size = window_size
        self.stride = stride
        self.horizons = horizons
        self.seq_len = seq_len
        self.samples = []

        increment_for_future_windows = int(window_size/stride)
        history_len = seq_len
        future_len = increment_for_future_windows + max(horizons) + 1
        sample_size = history_len + future_len

        df = pd.read_csv(os.path.join(dataset_path, csv_file_name))
        df = df[df['patient_id'].isin(patient_list)]
        df['file_name'] = df['patient_id'].astype(str) + '_' + df['episode_id'].astype(str) + '_' + df['segment_id'].astype(str) + '.npy'
        for patient in patient_list:
            patient_df = df[df['patient_id'] == patient]
            total_segments = len(patient_df)
            for i in range(0, total_segments - sample_size + 1, 1):
                rr_windows = []
                hrvs = []
                for j in range(i, i + sample_size, stride):
                    rr_window = np.load(os.path.join(dataset_path, patient_df.iloc[j]['file_name']))
                    rr_windows.append(rr_window)
                    hrv_features = patient_df.iloc[j][['RMSSD', 'pNN50', 'SDNN', 'alpha_1', 'sample_entropy', 'approximate_entropy']].values.astype(np.float32)
                    hrvs.append(hrv_features)
                self.samples.append((rr_windows, hrvs))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        rr_windows, hrvs = self.samples[idx]
        current_hrv = self.compute_hrv_metrics(rr_windows[-1])
        return torch.tensor(rr_windows, dtype=torch.float32), torch.tensor(hrvs, dtype=torch.float32), torch.tensor(current_hrv, dtype=torch.float32)

In [19]:
def get_loss_weights(epoch, total_epochs):
    """
    Gradually shifts focus from loss_1 (Horizon 4) to loss_4 (Horizon 16).
    """
    # Progressive factor from 0 to 1
    alpha = epoch / total_epochs 
    
    # Near horizon starts strong (1.0) and decays
    w1 = 1.2
    # Mid horizon stays relatively stable
    w2 = 1.0 
    # Far horizon starts low (0.2) and becomes the priority (1.5)
    w4 = 0.8
    
    return w1, w2, w4

In [20]:

def training_step(
    model, 
    rr_windows: torch.Tensor, 
    hrv_targets: torch.Tensor,
    epoch,
    current_hrv,
    total_epochs=EPOCHS
) -> torch.Tensor:
    """
    rr_windows: [B, T, W]  -> RR windows
    hrv_targets : [B, T, num_metrics] -> HRV targets for different horizons
    Returns:
        loss: scalar tensor
    """
    w1, w2, w4 = get_loss_weights(epoch,total_epochs)
    # Get context embeddings from model
    c_seq = model(rr_windows)  # [B, T, context_dim]
    last_context = c_seq[:, -1, :]  # [B, context_dim]
    y_pred_1, y_pred_2, y_pred_4 = model.predictor(last_context,current_hrv)  # Each: [B, num_metrics]
    y_true_1, y_true_2, y_true_4 = hrv_targets[:, 0, :], hrv_targets[:, 1, :], hrv_targets[:, 2, :]  # Each: [B, num_metrics]
    loss_1 = F.mse_loss(y_pred_1, y_true_1)
    loss_2 = F.mse_loss(y_pred_2, y_true_2)
    loss_4 = F.mse_loss(y_pred_4, y_true_4)
    total_loss = (w1 * loss_1) + (w2 * loss_2) + (w4 * loss_4)
    return total_loss,loss_1,loss_2,loss_4

In [21]:
def validation_step(model, dataloader, device):
    model.eval()
    val_loss = 0.0
    val_loss_1 = 0.0
    val_loss_2 = 0.0
    val_loss_4 = 0.0
    count = 0

    with torch.no_grad():
        for rr_windows, hrv_targets,current_hrv in dataloader:
            rr_windows = rr_windows.to(device)  # [B, 1, W] if single-channel
            # Move targets to device
            hrv_targets = hrv_targets.to(device)

            # Compute loss
            loss,loss_1,loss_2,loss_3 = training_step(model, rr_windows, hrv_targets,current_hrv,EPOCHS)
            val_loss += loss.item()
            val_loss_1 += loss_1.item()
            val_loss_2 += loss_2.item()
            val_loss_4 += loss_4.item()

            count += 1

    return (val_loss / count,val_loss_1 / count,val_loss_2 / count,val_loss_4 / count) if count > 0 else (0.0,0.0,0.0,0.0)

In [23]:
dataset_path = os.path.join('/kaggle','input','datasets','yasanthaniroshan','irida-200-rr')
df = pd.read_csv(os.path.join(dataset_path,'200x20_extracted_rr_intervals.csv'))
all_patients = np.sort(np.unique(df['patient_id'].values))
# print(all_patients)

In [24]:
np.random.seed(42)
np.random.shuffle(all_patients)

val_count = max(1, int(len(all_patients) * VALIDATION_RATIO))

val_patients = all_patients[:val_count]
train_patients = all_patients[val_count:]

# print(train_patients)
# print(val_patients)


In [ ]:
train_dataset = RRSequenceDataset(dataset_path,'200x20_extracted_rr_intervals.csv',train_patients, WINDOW_SIZE, STRIDE, HORIZONS, SEQUENCE_LENGTH)
val_dataset = RRSequenceDataset(dataset_path,'200x20_extracted_rr_intervals.csv',val_patients, WINDOW_SIZE, STRIDE, HORIZONS, SEQUENCE_LENGTH)
print(f"Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")

In [ ]:
rr_windows,hrvs,current_hrv = val_dataset[0]
print(hrvs,rr_windows,current_hrv)

In [ ]:
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model & optimizer
model = CPCPreModel().to(device)
optimizer = AdamW(model.parameters(), lr=LR)

In [ ]:
for epoch in range(1, EPOCHS+1):
    model.train()
    running_loss = 0.0
    batch_count = 0

    for rr_windows, hrv_targets,current_hrv in train_loader:
        rr_windows = rr_windows.to(device)
        hrv_targets = hrv_targets.to(device)
        optimizer.zero_grad()
        loss,loss_1,loss_2,loss_4 = training_step(model, rr_windows, hrv_targets,current_hrv,epoch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        batch_count += 1
        
    train_loss = running_loss / batch_count
    val_loss,val_loss_1,val_loss_2,val_loss_4 = validation_step(model, val_loader, device)
    run.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_loss_1": val_loss_1,
        "val_loss_2": val_loss_2,
        "val_loss_4": val_loss_4
    })
    print(f"Epoch {epoch:02d}: Train Loss = {train_loss:.6f} Validation Loss = {val_loss:.6f} 1 : {val_loss_1:.6f} 2 : {val_loss_2:.6f} 4 : {val_loss_4:.6f}")


In [ ]:
os.listdir('/kaggle/working')

In [ ]:
model_path = "cpc_pre_model.pt"
torch.save(model.state_dict(), model_path)

In [ ]:
artifact = wandb.Artifact(
    name="cpc-pre-model",
    type="model",
    description="CPC Pretraining Model"
)

artifact.add_file(model_path)
wandb.log_artifact(artifact)

In [ ]:
run.finish()

In [ ]:
class RRSequenceCoxDataset(Dataset):
    def __init__(
        self,
        patient_ids,          # list of main record names (e.g., ['p16', 'p18', ...])
        dataset_path,         # path to the .hea/.dat files
        window_size=50,
        stride=25,
        seq_len=10
    ):
        """
        patient_ids : list of main record names (e.g., even-numbered 'p16')
        dataset_path : folder containing .hea/.dat/.qrs files
        """
        self.samples = []

        for rec_name in patient_ids:
            # --- Paths to main and continuation records ---
            main_path = os.path.join(dataset_path, rec_name)
            cont_path = os.path.join(dataset_path, rec_name + 'c')

            rr_main = self.load_rr_intervals(main_path)
            rr_cont = self.load_rr_intervals(cont_path)

            if len(rr_main) == 0:
                continue

            # Concatenate main + continuation
            rr_seq = np.concatenate([rr_main, rr_cont]) if len(rr_cont) > 0 else rr_main

            total_len = len(rr_seq)
            history_len = window_size + stride * (seq_len - 1)
            total_windows = (total_len - window_size) // stride + 1

            # Index where continuation starts
            cont_start_idx = len(rr_main)

            for i in range(total_windows):
                start_idx = i * stride
                end_idx = start_idx + history_len
                if end_idx > total_len:
                    break

                rr_subseq = rr_seq[start_idx:end_idx]

                # --- Create RR windows ---
                rr_windows = []
                for j in range(seq_len):
                    w_start = j * stride
                    w_end = w_start + window_size
                    rr_windows.append(rr_subseq[w_start:w_end])
                rr_windows = np.stack(rr_windows)

                # --- Event & duration in minutes ---
                window_end_idx = start_idx + (seq_len - 1) * stride + window_size

                if window_end_idx <= cont_start_idx or len(rr_cont) == 0:
                    # Main record → censored
                    event_label = 0
                    # Duration = time to continuation in minutes
                    # Use mean RR of last stride
                    last_stride_start = start_idx + (seq_len - 1) * stride
                    last_stride_end = min(last_stride_start + stride, cont_start_idx)
                    if last_stride_start >= last_stride_end:
                        duration_min = 0.0
                    else:
                        mean_rr_ms = rr_seq[last_stride_start:last_stride_end].mean()
                        steps_to_event = (cont_start_idx - last_stride_start)
                        duration_min = mean_rr_ms * steps_to_event / 1000 / 60
                else:
                    # Window includes continuation → event occurred
                    event_label = 1
                    duration_min = 0.0

                if duration_min < 0:
                    duration_min = 0.0

                self.samples.append((rr_windows, duration_min, event_label))

    def load_rr_intervals(self, record_path):
        """Load RR intervals from a WFDB record"""
        try:
            record = wfdb.rdrecord(record_path)
            annotation = wfdb.rdann(record_path, 'qrs')
            rr_samples = np.diff(annotation.sample)
            rr_intervals = rr_samples / record.fs * 1000  # convert to ms
            return rr_intervals
        except Exception as e:
            print(f"Warning: Could not read RR for {record_path}: {e}")
            return []

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        rr_windows, duration_min, event = self.samples[idx]
        return (
            torch.tensor(rr_windows, dtype=torch.float32),  # (T, W)
            torch.tensor(duration_min, dtype=torch.float32),  # minutes
            torch.tensor(event, dtype=torch.float32)        # 0/1
        )


In [ ]:
cox_train_dataset = RRSequenceCoxDataset(train_patients,dataset_path)
cox_val_dataset = RRSequenceCoxDataset(val_patients, dataset_path)


In [ ]:
from torch.utils.data import DataLoader, Subset

# 1. Create a subset of the first 512 samples
indices = list(range(512))
train_subset = Subset(cox_train_dataset, indices)

In [ ]:
# Training loader
cox_train_loader = DataLoader(
    train_subset,
    batch_size=512,
    shuffle=True,   # shuffle windows across patients
    drop_last=True  # optional
)

# Validation loader
cox_val_loader = DataLoader(
    cox_val_dataset,
    batch_size=512,
    shuffle=False   # preserve order for validation
)

In [ ]:
for rr_windows, duration_min, event in cox_train_loader:
    print(rr_windows.shape)  # (B, T, W)
    print(duration_min.shape) # (B,)
    print(event.shape)        # (B,)
    break

In [ ]:
class CoxPHLoss(nn.Module):
    def forward(self, risk, duration, event):
        # 1. Sort by descending duration
        duration, idx = duration.sort(descending=True)
        risk = risk[idx]
        event = event[idx]

        # 2. Log-Sum-Exp trick for the denominator
        # This is more stable than log(cumsum(exp(risk)))
        # We use a reverse cumsum because for Cox, the risk set 
        # for subject i includes all j where duration[j] >= duration[i]
        
        # Since we sorted descending, the risk set for index i is indices 0...i
        # Wait, usually Cox risk sets are all j >= i. 
        # If duration is [10, 8, 5], risk set for 10 is {10}, for 8 is {10, 8}...
        # Actually, let's stick to the logic: Risk set = people alive at time T
        
        # Standard implementation with log_cumsum_exp:
        # We need the sum of exp(risk) for all those with duration >= current
        # Since sorted descending, that is the cumsum from index 0 to i.
        
        m, _ = torch.max(risk, dim=0, keepdim=True)
        risk_exp = torch.exp(risk - m)
        risk_cumsum = torch.cumsum(risk_exp, dim=0)
        log_cumsum = m.squeeze() + torch.log(risk_cumsum)

        # 3. Mask for events
        event_mask = event.bool()
        if not event_mask.any():
            return torch.tensor(0.0, requires_grad=True, device=risk.device)

        loss = -torch.mean(risk[event_mask] - log_cumsum[event_mask])
        return loss

In [ ]:
class CoxHead(nn.Module):
    def __init__(self, context_dim=CONTEXT_SIZE):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(context_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
    def forward(self, c_last):
        return self.net(c_last)


In [ ]:
class DeepSurvCox(nn.Module):
    def __init__(self, encoder, context, context_dim=CONTEXT_SIZE):
        super().__init__()
        self.encoder = encoder
        self.context = context
        self.cox_head = CoxHead(context_dim)

    def forward(self, rr_windows):
        B, T, W = rr_windows.shape
        z_list = []

        # REMOVE torch.no_grad() from here!
        for t in range(T):
            z_t = self.encoder(rr_windows[:, t, :])
            z_list.append(z_t)
        
        z_seq = torch.stack(z_list, dim=1)
        c_seq = self.context(z_seq)
        c_last = c_seq[:, -1, :]
        
        risk = self.cox_head(c_last).squeeze(-1)
        return risk

In [ ]:
def concordance_index(risk, duration, event):
    """
    Computes Harrell's C-index (vectorized, correct)
    risk: [N] tensor, higher = higher predicted risk
    duration: [N] tensor
    event: [N] tensor, 1=event occurred, 0=censored
    """
    risk = risk.detach().cpu().numpy()
    duration = duration.detach().cpu().numpy()
    event = event.detach().cpu().numpy()

    n = len(risk)
    if n <= 1:
        return 0.0

    # Expand to pairwise comparisons
    dur_i = duration[:, None]
    dur_j = duration[None, :]
    event_i = event[:, None]

    # Comparable pairs: i had event AND dur[i] < dur[j]
    comparable = (event_i == 1) & (dur_i < dur_j)

    # Count concordant pairs: risk[i] > risk[j]
    concordant = (risk[:, None] > risk[None, :]) & comparable
    tied = (risk[:, None] == risk[None, :]) & comparable

    num = concordant.sum() + 0.5 * tied.sum()
    den = comparable.sum()
    return num / den if den > 0 else 0.0


In [ ]:
# Assume encoder and context are already trained
encoder = model.encoder
context = model.context

# # Freeze parameters
# for param in encoder.parameters():
#     param.requires_grad = False

# for param in context.parameters():
#     param.requires_grad = False

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cox_model.to(device)
criterion = CoxPHLoss()
# Unfreeze the context layer
for param in cox_model.context.parameters():
    param.requires_grad = True

# Update optimizer to include context params
optimizer = torch.optim.AdamW([
    {'params': cox_model.context.parameters(), 'lr': 1e-5}, # Slower for context
    {'params': cox_model.cox_head.parameters(), 'lr': 1e-3}  # Faster for the head
])

In [ ]:
for epoch in range(1, 250 + 1):
    # --- Training ---
    cox_model.train()
    running_loss = 0.0
    batch_count = 0

    for rr_windows, duration_min, event in cox_train_loader:
        rr_windows = rr_windows.to(device)       # [B, T, W]
        duration_min = duration_min.to(device)   # [B]
        event = event.to(device)                 # [B]

        optimizer.zero_grad()
        risk = cox_model(rr_windows)            # [B]
        loss = criterion(risk, duration_min, event)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        batch_count += 1

    avg_train_loss = running_loss / batch_count

    # --- Validation ---
    cox_model.eval()
    val_loss = 0.0
    val_batches = 0
    all_risk = []
    all_duration = []
    all_event = []

    with torch.no_grad():
        for rr_windows, duration_min, event in cox_val_loader:
            rr_windows = rr_windows.to(device)
            duration_min = duration_min.to(device) / 60.0  # normalize to hours
            event = event.to(device)

            risk = cox_model(rr_windows)
            loss = criterion(risk, duration_min, event)

            val_loss += loss.item()
            val_batches += 1

            all_risk.append(risk)
            all_duration.append(duration_min)
            all_event.append(event)

    avg_val_loss = val_loss / val_batches

    # Concatenate all validation tensors
    all_risk = torch.cat(all_risk)
    all_duration = torch.cat(all_duration)
    all_event = torch.cat(all_event)

    c_index = concordance_index(all_risk, all_duration, all_event)

    print(f"Epoch {epoch:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val C-index: {c_index:.4f}")
